# Fraud Detection with Cost-Sensitive Learning (IEEE-CIS and ULB)

This notebook contains the full pipeline for the thesis: preprocessing with a three-way split, baseline models, the cost-sensitive experiments (weighted training and threshold moving), and the evaluation.

The cost model is taken from Bahnsen et al. (2013): a missed fraud (FN) costs the transaction amount, a false alarm (FP) and a caught fraud (TP) both cost a fixed administrative cost $C_a$, and a correctly approved transaction (TN) costs nothing. The decision theory behind the thresholds follows Elkan (2001).


## 1. Setup & Data Loading

In [1]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/azizhassine2025/fraud-detection-bachelorarbeit/creditcard-ULB .csv
/kaggle/input/datasets/azizhassine2025/fraud-detection-bachelorarbeit/train_identity.csv
/kaggle/input/datasets/azizhassine2025/fraud-detection-bachelorarbeit/train_transaction.csv


In [2]:
import pandas as pd

In [3]:
base_path = "/kaggle/input/datasets/azizhassine2025/fraud-detection-bachelorarbeit"

df_identity = pd.read_csv(f"{base_path}/train_identity.csv") #IEEE-CIS identity dataset
df_transactions = pd.read_csv(f"{base_path}/train_transaction.csv") #IEEE-CIS transaction dataset
df_ulm = pd.read_csv(f"{base_path}/creditcard-ULB .csv") # ULB dataset

## 2. Dataset Exploration

### 2.1 Basic Structure

In [4]:
df_identity.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


In [5]:
df_transactions.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
df_ulm.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [7]:
df_identity.shape

(144233, 41)

In [8]:
df_transactions.shape

(590540, 394)

In [9]:
df_ulm.shape

(284807, 31)

In [10]:
print("Unique TransactionIDs (transactions):", df_transactions['TransactionID'].nunique())
print("Unique TransactionIDs (identity):", df_identity['TransactionID'].nunique())

Unique TransactionIDs (transactions): 590540
Unique TransactionIDs (identity): 144233


### 2.2 Class Imbalance

In [11]:
counts = df_transactions['isFraud'].value_counts()
counts

isFraud
0    569877
1     20663
Name: count, dtype: int64

In [12]:
class_percent = df_transactions["isFraud"].value_counts(normalize=True) * 100
class_percent

isFraud
0    96.500999
1     3.499001
Name: proportion, dtype: float64

In [13]:
counts_ulm = df_ulm['Class'].value_counts()
counts_ulm

Class
0    284315
1       492
Name: count, dtype: int64

In [14]:
class_ulm_percent = df_ulm["Class"].value_counts(normalize=True) * 100
class_ulm_percent

Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64

### 2.3 Duplicate Rows (ULB)

In [15]:
df_ulm.duplicated().sum()

np.int64(1081)

**Finding.** There are 1,081 duplicate rows. These are treated as data artifacts and removed in Section 3.2c.

### 2.4 Transaction Amount Distributions

In [16]:
fraud = df_transactions[df_transactions['isFraud'] == 1]
ulm_fraud = df_ulm[df_ulm['Class'] == 1]

In [17]:
print("IEEE-CIS — all transactions:")
print(df_transactions['TransactionAmt'].describe())
print("\nIEEE-CIS — fraud only:")
print(fraud['TransactionAmt'].describe())

IEEE-CIS — all transactions:
count    590540.000000
mean        135.027176
std         239.162522
min           0.251000
25%          43.321000
50%          68.769000
75%         125.000000
max       31937.391000
Name: TransactionAmt, dtype: float64

IEEE-CIS — fraud only:
count    20663.000000
mean       149.244779
std        232.212163
min          0.292000
25%         35.044000
50%         75.000000
75%        161.000000
max       5191.000000
Name: TransactionAmt, dtype: float64


In [18]:
print("ULB — all transactions:")
print(df_ulm['Amount'].describe())
print("\nULB — fraud only:")
print(ulm_fraud['Amount'].describe())

ULB — all transactions:
count    284807.000000
mean         88.349619
std         250.120109
min           0.000000
25%           5.600000
50%          22.000000
75%          77.165000
max       25691.160000
Name: Amount, dtype: float64

ULB — fraud only:
count     492.000000
mean      122.211321
std       256.683288
min         0.000000
25%         1.000000
50%         9.250000
75%       105.890000
max      2125.870000
Name: Amount, dtype: float64


### 2.5 Missing Values Overview (descriptive)

In [19]:
missing_trans = df_transactions.isnull().mean() * 100
print(missing_trans.sort_values(ascending=False).head(175))

dist2    93.628374
D7       93.409930
D13      89.509263
D14      89.469469
D12      89.041047
           ...    
M7       58.635317
M8       58.633115
M9       58.633115
D5       52.467403
M4       47.658753
Length: 175, dtype: float64


In [20]:
missing_identity = df_identity.isnull().mean() * 100
print(missing_identity.sort_values(ascending=False).head(13))

id_24    96.708798
id_25    96.441868
id_07    96.425922
id_08    96.425922
id_21    96.423149
id_26    96.420375
id_23    96.416215
id_27    96.416215
id_22    96.416215
id_18    68.722137
id_04    54.016071
id_03    54.016071
id_33    49.187079
dtype: float64


In [21]:
missing_ulm = df_ulm.isnull().mean() * 100
print(missing_ulm.sort_values(ascending=False).head(20))

Time    0.0
V1      0.0
V2      0.0
V3      0.0
V4      0.0
V5      0.0
V6      0.0
V7      0.0
V8      0.0
V9      0.0
V10     0.0
V11     0.0
V12     0.0
V13     0.0
V14     0.0
V15     0.0
V16     0.0
V17     0.0
V18     0.0
V19     0.0
dtype: float64


### 2.6 Feature–Fraud Relationships

In [22]:
df_merge_raw = df_transactions.merge(df_identity, on='TransactionID', how='left')

In [23]:
# Fraud rate per ProductCD
print(df_merge_raw.groupby('ProductCD')['isFraud'].mean().sort_values(ascending=False))

# Fraud rate per DeviceType
print(df_merge_raw.groupby('DeviceType')['isFraud'].mean().sort_values(ascending=False))

# Fraud rate per card4 (visa/mastercard etc.)
print(df_merge_raw.groupby('card4')['isFraud'].mean().sort_values(ascending=False))

ProductCD
C    0.116873
S    0.058996
H    0.047662
R    0.037826
W    0.020399
Name: isFraud, dtype: float64
DeviceType
mobile     0.101662
desktop    0.065215
Name: isFraud, dtype: float64
card4
discover            0.077282
visa                0.034756
mastercard          0.034331
american express    0.028698
Name: isFraud, dtype: float64


In [24]:
# Fraud rate per card6 (credit vs debit)
print(df_merge_raw.groupby('card6')['isFraud'].mean().sort_values(ascending=False))

# Top 10 most fraudulent email domains
print(df_merge_raw.groupby('P_emaildomain')['isFraud'].mean().sort_values(ascending=False).head(10))

card6
credit             0.066785
debit              0.024263
charge card        0.000000
debit or credit    0.000000
Name: isFraud, dtype: float64
P_emaildomain
protonmail.com    0.407895
mail.com          0.189624
outlook.es        0.130137
aim.com           0.126984
outlook.com       0.094584
hotmail.es        0.065574
live.com.mx       0.054740
hotmail.com       0.052950
gmail.com         0.043542
yahoo.fr          0.034965
Name: isFraud, dtype: float64


In [25]:
df_merge_raw['has_identity'] = df_merge_raw['id_01'].notna().astype(int)
print(df_merge_raw.groupby('has_identity')['isFraud'].mean())

has_identity
0    0.020939
1    0.078470
Name: isFraud, dtype: float64


In [26]:
df_with_identity = df_transactions.merge(df_identity, on='TransactionID', how='inner')

print(f"Transactions with identity: {len(df_with_identity)}")
print(f"Fraud: {df_with_identity['isFraud'].sum()}")
print(f"Fraud rate: {df_with_identity['isFraud'].mean():.2%}")

Transactions with identity: 144233
Fraud: 11318
Fraud rate: 7.85%


In [27]:
fraud_df = fraud.merge(df_identity, on='TransactionID', how='left')
len(fraud_df)

20663

**Main findings from 2.6:**

- Transactions that come with an identity record have a much higher fraud rate (about 7.85%) than those without one (about 2.09%).
- The email domain `protonmail.com` stands out with a fraud rate of roughly 40.8%.
- Product C and credit cards (rather than debit) are higher risk, and mobile transactions are slightly riskier than desktop.

## 3. Preprocessing

### 3.1 Missing Values: Informative-Missingness Check (Transactions)

The usual approach is to drop any column with more than 50% missing values. Madley-Dowd et al. (2019) argue that the proportion of missing values on its own is not a good reason to drop a column; what matters is whether the missingness itself says something about the outcome. Following that idea, for each column with more than 50% missing values I compare the fraud rate between the rows where the value is missing and the rows where it is present.

In [28]:
high_missing_cols = missing_trans[missing_trans > 50].index

base_rate = df_transactions['isFraud'].mean() * 100

results = []
for col in high_missing_cols:
    fraud_rate_missing = df_transactions.loc[df_transactions[col].isna(), 'isFraud'].mean() * 100
    fraud_rate_present = df_transactions.loc[df_transactions[col].notna(), 'isFraud'].mean() * 100
    gap = abs(fraud_rate_missing - fraud_rate_present)
    relative_gap = gap / base_rate
    results.append({
        'col': col,
        'fraud_rate_missing': fraud_rate_missing,
        'fraud_rate_present': fraud_rate_present,
        'gap': gap,
        'relative_gap': relative_gap
    })

gap_df = pd.DataFrame(results).sort_values('relative_gap', ascending=False)
print(gap_df)

     col  fraud_rate_missing  fraud_rate_present        gap  relative_gap
5     D7            2.696226           14.877817  12.181591      3.481448
8    D12            2.484676           11.740346   9.255670      2.645232
10   D14            2.545647           11.598887   9.053241      2.587379
9    D13            2.615648           11.035963   8.420316      2.406491
4     D6            2.502160           10.545589   8.043429      2.298779
..   ...                 ...                 ...        ...           ...
17  V140            3.345098            4.454207   1.109109      0.316979
35  V158            3.345098            4.454207   1.109109      0.316979
34  V157            3.345098            4.454207   1.109109      0.316979
3     D5            3.157426            3.876038   0.718612      0.205376
11    M5            3.735998            3.152988   0.583010      0.166622

[174 rows x 5 columns]


**Finding.** The columns fall into three groups by relative gap: weak (about 0.11 to 0.32), moderate (about 0.72 to 1.96), and strong (about 2.27 to 3.48, the D-block D6 to D14). `D7` on its own has a fraud rate of 2.7% when it is missing versus 14.9% when it is present.

### 3.2 Missing Values: Flag-and-Drop Decision (Transactions)

The rule is: any column whose relative gap is at least 1.0 (its fraud rate differs by at least 100% relative to the baseline) gets a binary `{col}_was_missing` indicator, and then the original column is dropped. Columns below that threshold are dropped without an indicator, because their missingness carries no useful signal. I checked the threshold against 0.5 and 2.0 as well, and settled on 1.0 as a clear and defensible cutoff.

In [29]:
gap_threshold = 1.0

cols_to_flag = gap_df[gap_df['relative_gap'] >= gap_threshold]['col'].tolist()
cols_to_drop_clean = gap_df[gap_df['relative_gap'] < gap_threshold]['col'].tolist()

# Only touch columns that still exist, so the cell can be re-run safely.
cols_to_flag = [c for c in cols_to_flag if c in df_transactions.columns]
cols_to_drop_clean = [c for c in cols_to_drop_clean if c in df_transactions.columns]

print(f"Flag (informative missingness): {len(cols_to_flag)}")
print(f"Drop without a flag (no signal): {len(cols_to_drop_clean)}")

if cols_to_flag:
    flags = df_transactions[cols_to_flag].isna().astype(int)
    flags.columns = [f"{c}_was_missing" for c in cols_to_flag]
    df_transactions = pd.concat([df_transactions, flags], axis=1)

df_transactions = df_transactions.drop(columns=cols_to_flag + cols_to_drop_clean, errors='ignore')
print("New transactions shape:", df_transactions.shape)

Flag (informative missingness): 121
Drop without a flag (no signal): 53
New transactions shape: (590540, 341)


### 3.1b Missing Values: Identity Table

`df_identity` has no `isFraud` column of its own, so for this analysis only I merge it with the target. This is read-only; the real `df_identity` is not changed at this step. I then apply the same gap check from 3.1 and 3.2 to the identity columns with high missingness. The identity table is much smaller (144,233 rows), so the missing and present groups can be small, and the group sizes are printed as a check.

In [30]:
# Attach isFraud to the identity data just for this gap analysis.
# This is read-only and does not change df_identity.
identity_with_target = df_identity.merge(
    df_transactions[['TransactionID', 'isFraud']],
    on='TransactionID',
    how='left'
)

In [31]:
high_missing_cols_identity = missing_identity[missing_identity > 50].index

base_rate_identity = identity_with_target['isFraud'].mean() * 100

results_identity = []
for col in high_missing_cols_identity:
    n_missing = identity_with_target[col].isna().sum()
    n_present = identity_with_target[col].notna().sum()
    fraud_rate_missing = identity_with_target.loc[identity_with_target[col].isna(), 'isFraud'].mean() * 100
    fraud_rate_present = identity_with_target.loc[identity_with_target[col].notna(), 'isFraud'].mean() * 100
    gap = abs(fraud_rate_missing - fraud_rate_present)
    relative_gap = gap / base_rate_identity
    results_identity.append({
        'col': col,
        'n_missing': n_missing,
        'n_present': n_present,
        'fraud_rate_missing': fraud_rate_missing,
        'fraud_rate_present': fraud_rate_present,
        'gap': gap,
        'relative_gap': relative_gap
    })

gap_df_identity = pd.DataFrame(results_identity).sort_values('relative_gap', ascending=False)
print(gap_df_identity)

      col  n_missing  n_present  fraud_rate_missing  fraud_rate_present  \
0   id_03      77909      66324            5.398606           10.723117   
1   id_04      77909      66324            5.398606           10.723117   
8   id_24     139486       4747            7.825875            8.468506   
4   id_18      99120      45113            7.650323            8.279210   
3   id_08     139078       5155            7.831577            8.263822   
2   id_07     139078       5155            7.831577            8.263822   
5   id_21     139074       5159            7.831802            8.257414   
6   id_22     139064       5169            7.832365            8.241439   
7   id_23     139064       5169            7.832365            8.241439   
11  id_27     139064       5169            7.832365            8.241439   
10  id_26     139070       5163            7.832746            8.231648   
9   id_25     139101       5132            7.836752            8.125487   

         gap  relative_g

In [32]:
gap_threshold = 1.0

cols_to_flag_identity = gap_df_identity[gap_df_identity['relative_gap'] >= gap_threshold]['col'].tolist()
cols_to_drop_clean_identity = gap_df_identity[gap_df_identity['relative_gap'] < gap_threshold]['col'].tolist()

# Only touch columns that still exist, so the cell can be re-run safely.
cols_to_flag_identity = [c for c in cols_to_flag_identity if c in df_identity.columns]
cols_to_drop_clean_identity = [c for c in cols_to_drop_clean_identity if c in df_identity.columns]

print(f"Flag (informative missingness): {len(cols_to_flag_identity)}")
print(f"Drop without a flag (no signal): {len(cols_to_drop_clean_identity)}")

if cols_to_flag_identity:
    flags_identity = df_identity[cols_to_flag_identity].isna().astype(int)
    flags_identity.columns = [f"{c}_was_missing" for c in cols_to_flag_identity]
    df_identity = pd.concat([df_identity, flags_identity], axis=1)

df_identity = df_identity.drop(columns=cols_to_flag_identity + cols_to_drop_clean_identity, errors='ignore')
print("New identity shape:", df_identity.shape)

Flag (informative missingness): 0
Drop without a flag (no signal): 12
New identity shape: (144233, 29)


### 3.2b Merging the Cleaned Datasets

Both `df_transactions` and `df_identity` have now had their missing-value treatment. This is the single, final merge that everything downstream uses (encoding, scaling, splitting).

In [33]:
df = df_transactions.merge(df_identity, on='TransactionID', how='left')
print("Merged shape:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())

Merged shape: (590540, 369)
Remaining missing values: 27130366


After the left join, missing values remain: the roughly 76% of transactions without an identity record show NaN across all identity columns. This is expected (these are the `has_identity` = 0 group) and is handled in Section 3.4.

In [34]:
df['has_identity'] = df['id_01'].notna().astype(int)

print(df['has_identity'].value_counts())
print(df.groupby('has_identity')['isFraud'].mean())

has_identity
0    446307
1    144233
Name: count, dtype: int64
has_identity
0    0.020939
1    0.078470
Name: isFraud, dtype: float64


### 3.2c Removing Duplicates (ULB)

Applying the finding from Section 2.3.

In [35]:
print("Before:", df_ulm.shape)
df_ulm = df_ulm.drop_duplicates()
print("After:", df_ulm.shape)

Before: (284807, 31)
After: (283726, 31)


### 3.3 Encoding Preparation: Cardinality and Feature Types

Here I work on the final merged `df` (not a separate rebuild) to tell categorical and numeric columns apart before encoding.

In [36]:
cardinality_summary = df.nunique()
print(cardinality_summary.sort_values(ascending=True))

num_low_cardinality = (cardinality_summary <= 10).sum()
print(f"Number of columns with 10 or fewer unique values: {num_low_cardinality}")

has_identity          2
isFraud               2
DeviceType            2
id_38                 2
id_37                 2
                  ...  
V127              24414
V307              37367
id_02            115655
TransactionDT    573349
TransactionID    590540
Length: 370, dtype: int64
Number of columns with 10 or fewer unique values: 227


In [37]:
cardinality_summary_ulm = df_ulm.nunique()
print(cardinality_summary_ulm.sort_values(ascending=True))

num_low_cardinality_ulm = (cardinality_summary_ulm <= 10).sum()
print(f"Number of columns with 10 or fewer unique values: {num_low_cardinality_ulm}")

Class          2
Amount     32767
Time      124592
V1        275663
V4        275663
V5        275663
V2        275663
V3        275663
V8        275663
V9        275663
V10       275663
V11       275663
V12       275663
V13       275663
V6        275663
V7        275663
V15       275663
V14       275663
V16       275663
V17       275663
V20       275663
V21       275663
V18       275663
V19       275663
V23       275663
V22       275663
V24       275663
V25       275663
V27       275663
V26       275663
V28       275663
dtype: int64
Number of columns with 10 or fewer unique values: 1


In [38]:
low_card_cols = cardinality_summary[cardinality_summary <= 10].index.tolist()

numeric_cols = [c for c in low_card_cols if df[c].dtype in ['float64', 'int64']]
categorical_cols = [c for c in low_card_cols if df[c].dtype == 'object']

print(f"Numeric (no encoding needed): {len(numeric_cols)}")
print(f"Categorical (needs encoding): {len(categorical_cols)}")
print()
print("Categorical columns:", categorical_cols)

Numeric (no encoding needed): 208
Categorical (needs encoding): 19

Categorical columns: ['ProductCD', 'card4', 'card6', 'M1', 'M2', 'M3', 'M4', 'M6', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType']


In [39]:
low_card_cols_ulm = cardinality_summary_ulm[cardinality_summary_ulm <= 10].index.tolist()

numeric_cols_ulm = [c for c in low_card_cols_ulm if df_ulm[c].dtype in ['float64', 'int64']]
categorical_cols_ulm = [c for c in low_card_cols_ulm if df_ulm[c].dtype == 'object']

print(f"Numeric (no encoding needed): {len(numeric_cols_ulm)}")
print(f"Categorical (needs encoding): {len(categorical_cols_ulm)}")
print()
print("Categorical columns:", categorical_cols_ulm)

Numeric (no encoding needed): 1
Categorical (needs encoding): 0

Categorical columns: []


In [40]:
for col in categorical_cols:
    print(col, df[col].value_counts(dropna=False))
    print()

ProductCD ProductCD
W    439670
C     68519
R     37699
H     33024
S     11628
Name: count, dtype: int64

card4 card4
visa                384767
mastercard          189217
american express      8328
discover              6651
NaN                   1577
Name: count, dtype: int64

card6 card6
debit              439938
credit             148986
NaN                  1571
debit or credit        30
charge card            15
Name: count, dtype: int64

M1 M1
T      319415
NaN    271100
F          25
Name: count, dtype: int64

M2 M2
T      285468
NaN    271100
F       33972
Name: count, dtype: int64

M3 M3
NaN    271100
T      251731
F       67709
Name: count, dtype: int64

M4 M4
NaN    281444
M0     196405
M2      59865
M1      52826
Name: count, dtype: int64

M6 M6
F      227856
T      193324
NaN    169360
Name: count, dtype: int64

id_12 id_12
NaN         446307
NotFound    123025
Found        21208
Name: count, dtype: int64

id_15 id_15
NaN        449555
Found       67728
New         61612

### 3.4 Categorical Imputation (constant fill)

Missing categorical values are filled with an explicit `'Unknown'` category. Because this is a fixed value that uses no statistics from the data, it cannot leak information and is safe to apply before the train/test split. Missing numeric values are left as they are for now and imputed after the split, using training-set medians only (Section 3.7).

In [41]:
# Filling with the constant 'Unknown' uses no statistics from the data,
# so it is safe to do before the split.
# Numeric NaN are left for now and imputed after the split, using train medians only.
cat_cols_all = df.select_dtypes(include=['object']).columns
df[cat_cols_all] = df[cat_cols_all].fillna('Unknown')

cat_cols_all_ulm = df_ulm.select_dtypes(include=['object']).columns
df_ulm[cat_cols_all_ulm] = df_ulm[cat_cols_all_ulm].fillna('Unknown')

print("Categorical NaN filled with 'Unknown'.")

Categorical NaN filled with 'Unknown'.


### 3.5 Encoding

One-hot encoding of the low-cardinality categorical columns found in Section 3.3. The encoding uses no target or test-set information, so it cannot leak and is applied before the split; doing it here also guarantees that the training and test sets end up with the same dummy columns. High-cardinality columns such as `P_emaildomain` are left unencoded and handled at the modelling stage.

In [42]:
df = pd.get_dummies(df, columns=categorical_cols, drop_first=False)
print("Shape after encoding:", df.shape)

Shape after encoding: (590540, 417)


### 3.6 Train/Test Split

A stratified 80/20 split that keeps the fraud ratio the same in both parts. The split happens before imputation and scaling, so nothing from the test set can influence the training pipeline (see Sections 3.7 and 3.8).

In [43]:
from sklearn.model_selection import train_test_split

# Three-way split: train / validation / test (stratified, seeded).
#   - validation is used only to choose the cost-optimal thresholds
#   - test is used once, for the final numbers
# Splitting before any fitting is what keeps the pipeline free of leakage.

# ---------- IEEE-CIS ----------
X = df.drop(columns=['isFraud', 'TransactionID'])
y = df['isFraud']

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.20, stratify=y_trainval, random_state=42
)

# ---------- ULB ----------
X_ulm = df_ulm.drop(columns=['Class'])
y_ulm = df_ulm['Class']

X_ulm_trainval, X_ulm_test, y_ulm_trainval, y_ulm_test = train_test_split(
    X_ulm, y_ulm, test_size=0.20, stratify=y_ulm, random_state=42
)
X_ulm_train, X_ulm_val, y_ulm_train, y_ulm_val = train_test_split(
    X_ulm_trainval, y_ulm_trainval, test_size=0.20, stratify=y_ulm_trainval, random_state=42
)

print("IEEE  train/val/test:", X_train.shape, X_val.shape, X_test.shape)
print("ULB   train/val/test:", X_ulm_train.shape, X_ulm_val.shape, X_ulm_test.shape)
print("IEEE fraud rate train/val/test:",
      round(y_train.mean(), 5), round(y_val.mean(), 5), round(y_test.mean(), 5))
print("ULB  fraud rate train/val/test:",
      round(y_ulm_train.mean(), 5), round(y_ulm_val.mean(), 5), round(y_ulm_test.mean(), 5))

IEEE  train/val/test: (377945, 415) (94487, 415) (118108, 415)
ULB   train/val/test: (181584, 30) (45396, 30) (56746, 30)
IEEE fraud rate train/val/test: 0.03499 0.03499 0.03499
ULB  fraud rate train/val/test: 0.00166 0.00167 0.00167


### 3.7 Numeric Imputation (training medians only)

Missing numeric values are filled with the median computed on the training set only, and that same median is applied to the other partitions. The test set never contributes to the medians, so no test information reaches the training data. (ULB has no missing values, so this does nothing for it, but I run it the same way for consistency.)

In [44]:
# Medians are computed on the training set only, then applied to train / val / test.
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns
train_medians = X_train[num_cols].median()
X_train[num_cols] = X_train[num_cols].fillna(train_medians)
X_val[num_cols]   = X_val[num_cols].fillna(train_medians)
X_test[num_cols]  = X_test[num_cols].fillna(train_medians)

num_cols_ulm = X_ulm_train.select_dtypes(include=['float64', 'int64']).columns
train_medians_ulm = X_ulm_train[num_cols_ulm].median()
X_ulm_train[num_cols_ulm] = X_ulm_train[num_cols_ulm].fillna(train_medians_ulm)
X_ulm_val[num_cols_ulm]   = X_ulm_val[num_cols_ulm].fillna(train_medians_ulm)
X_ulm_test[num_cols_ulm]  = X_ulm_test[num_cols_ulm].fillna(train_medians_ulm)

print("Remaining NaN IEEE train/val/test:",
      X_train.isna().sum().sum(), X_val.isna().sum().sum(), X_test.isna().sum().sum())
print("Remaining NaN ULB  train/val/test:",
      X_ulm_train.isna().sum().sum(), X_ulm_val.isna().sum().sum(), X_ulm_test.isna().sum().sum())

Remaining NaN IEEE train/val/test: 0 0 0
Remaining NaN ULB  train/val/test: 0 0 0


### 3.7b Save Raw Transaction Amounts (before scaling)

In [45]:
# Save the real euro amounts before scaling replaces them with z-scores.
# The Bahnsen cost model needs the real amount for:
#   - the false-negative cost (a missed fraud costs the transaction amount)
#   - the example-dependent sample weights
#   - the amount-dependent threshold p*(x) = Ca / amount
# The index is kept, so these line up with y_* row for row.
amount_train     = X_train['TransactionAmt'].copy()
amount_val       = X_val['TransactionAmt'].copy()
amount_test      = X_test['TransactionAmt'].copy()

amount_ulm_train = X_ulm_train['Amount'].copy()
amount_ulm_val   = X_ulm_val['Amount'].copy()
amount_ulm_test  = X_ulm_test['Amount'].copy()

print("Saved raw amounts.")
print("IEEE test amount  min/median/max:",
      round(amount_test.min(), 2), round(amount_test.median(), 2), round(amount_test.max(), 2))
print("ULB  test amount  min/median/max:",
      round(amount_ulm_test.min(), 2), round(amount_ulm_test.median(), 2), round(amount_ulm_test.max(), 2))

Saved raw amounts.
IEEE test amount  min/median/max: 0.29 68.5 6085.23
ULB  test amount  min/median/max: 0.0 21.59 25691.16


### 3.8 Feature Scaling (fit on training only)

Standardization is needed for the neural network; XGBoost is not affected by feature scale. The `StandardScaler` is fit on the training set only and then used to transform both the training and test sets, so the mean and standard deviation used for scaling never see the test data.

In [46]:
from sklearn.preprocessing import StandardScaler

# IEEE-CIS: fit on train only, then transform train / val / test.
scale_cols = X_train.select_dtypes(include=['float64', 'int64']).columns.tolist()
scaler = StandardScaler().fit(X_train[scale_cols])
X_train[scale_cols] = scaler.transform(X_train[scale_cols])
X_val[scale_cols]   = scaler.transform(X_val[scale_cols])
X_test[scale_cols]  = scaler.transform(X_test[scale_cols])

# ULB: fit on train only.
scale_cols_ulm = X_ulm_train.select_dtypes(include=['float64', 'int64']).columns.tolist()
scaler_ulm = StandardScaler().fit(X_ulm_train[scale_cols_ulm])
X_ulm_train[scale_cols_ulm] = scaler_ulm.transform(X_ulm_train[scale_cols_ulm])
X_ulm_val[scale_cols_ulm]   = scaler_ulm.transform(X_ulm_val[scale_cols_ulm])
X_ulm_test[scale_cols_ulm]  = scaler_ulm.transform(X_ulm_test[scale_cols_ulm])

print("Scaling complete (fit on train only). IEEE cols:", len(scale_cols),
      "| ULB cols:", len(scale_cols_ulm))

Scaling complete (fit on train only). IEEE cols: 344 | ULB cols: 30


## 5. Cost-Sensitive Framework

### 5.1 Cost model, metrics, thresholds, calibration

The Bahnsen et al. (2013) cost matrix (these are costs, so lower is better):

| | actual legit | actual fraud |
|---|---|---|
| **predict legit** | 0 | amount |
| **predict fraud** | $C_a$ | $C_a$ |

So a missed fraud costs the transaction amount, a false alarm and a caught fraud both cost the fixed administrative cost $C_a$, and a correctly approved transaction is free. $C_a$ is swept over {1, 2.5, 5} EUR (2.5 is the primary value) as a sensitivity analysis. **Savings** = 1 − cost(model) / cost(approve-all).

In [47]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, average_precision_score, confusion_matrix
from sklearn.isotonic import IsotonicRegression

# Administrative-cost sweep for the sensitivity analysis; the primary value is 2.5 EUR.
CA_VALUES  = [1.0, 2.5, 5.0]
CA_PRIMARY = 2.5

# ---- Bahnsen cost model -------------------------------------------------
def total_cost(y_true, y_pred, amount, Ca):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred); amount = np.asarray(amount, dtype=float)
    fn = (y_pred == 0) & (y_true == 1)   # missed fraud: lose the amount
    fp = (y_pred == 1) & (y_true == 0)   # false alarm: costs Ca
    tp = (y_pred == 1) & (y_true == 1)   # caught fraud: costs Ca (investigation)
    return amount[fn].sum() + Ca * (fp.sum() + tp.sum())

def cost_approve_all(y_true, amount):
    # The "approve everything" baseline: every fraud becomes a false negative.
    y_true = np.asarray(y_true); amount = np.asarray(amount, dtype=float)
    return amount[y_true == 1].sum()

def savings(y_true, y_pred, amount, Ca):
    base = cost_approve_all(y_true, amount)
    return 0.0 if base == 0 else 1.0 - total_cost(y_true, y_pred, amount, Ca) / base

def evaluate(y_true, proba, amount, Ca, threshold=0.5, y_pred=None):
    if y_pred is None:
        y_pred = (np.asarray(proba) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'PR_AUC':    round(average_precision_score(y_true, proba), 4),
        'FN':        int(fn),
        'FP':        int(fp),
        'Cost':      round(float(total_cost(y_true, y_pred, amount, Ca)), 2),
        'Savings':   round(float(savings(y_true, y_pred, amount, Ca)), 4),
    }

# ---- decision rules -----------------------------------------------------
def cost_optimal_threshold(y_true, proba, amount, Ca, grid=1001):
    """Global threshold that minimises total cost, chosen on the validation set."""
    ts = np.linspace(0.0, 1.0, grid)
    best_t, best_c = 0.5, np.inf
    for t in ts:
        c = total_cost(y_true, (np.asarray(proba) >= t).astype(int), amount, Ca)
        if c < best_c:
            best_c, best_t = c, t
    return best_t

def amount_threshold_predict(proba, amount, Ca):
    """Elkan/Bahnsen rule for the Bahnsen matrix: flag fraud if p >= Ca/amount."""
    amount = np.asarray(amount, dtype=float)
    thr = np.where(amount > 0, Ca / amount, np.inf)   # very small or zero amounts are never worth declining
    thr = np.clip(thr, 0.0, 1.0)
    return (np.asarray(proba) >= thr).astype(int)

# ---- probability calibration (needed after reweighting; Dal Pozzolo 2015) ----
def calibrate(proba_val, y_val, proba_test):
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(np.asarray(proba_val), np.asarray(y_val))
    return iso.transform(np.asarray(proba_val)), iso.transform(np.asarray(proba_test))

print("Cost model, metrics, thresholds and calibration ready.  CA_VALUES =", CA_VALUES)

Cost model, metrics, thresholds and calibration ready.  CA_VALUES = [1.0, 2.5, 5.0]


In [48]:
# Experiment runner.
# For each model variant and each Ca, three decision rules are applied:
#   (a) default threshold 0.5
#   (b) cost-optimal global threshold t*, chosen on validation
#   (c) amount-dependent threshold p* = Ca / amount (Bayes minimum risk)
# The weighted variants have their probabilities recalibrated on validation first.

def run_all(tag, variants, y_val, y_test, amount_val, amount_test, ca_values=CA_VALUES):
    """
    variants: dict  name -> {'pv': val_proba, 'pt': test_proba, 'weighted': bool}
    Returns a tidy results DataFrame.
    """
    prepared = {}
    for name, info in variants.items():
        pv, pt = np.asarray(info['pv']), np.asarray(info['pt'])
        if info['weighted']:
            pv, pt = calibrate(pv, y_val, pt)     # recalibrate the distorted probabilities
        prepared[name] = (pv, pt)

    rows = []
    for Ca in ca_values:
        for name, (pv, pt) in prepared.items():
            # (a) t = 0.5
            rows.append({'Dataset_Model': tag, 'Variant': name, 'Ca': Ca,
                         'Rule': 't=0.5',
                         **evaluate(y_test, pt, amount_test, Ca, threshold=0.5)})
            # (b) cost-optimal global threshold from validation
            t = cost_optimal_threshold(y_val, pv, amount_val, Ca)
            rows.append({'Dataset_Model': tag, 'Variant': name, 'Ca': Ca,
                         'Rule': f't*={t:.3f}',
                         **evaluate(y_test, pt, amount_test, Ca, threshold=t)})
            # (c) amount-dependent threshold Ca/amount
            yp = amount_threshold_predict(pt, amount_test, Ca)
            rows.append({'Dataset_Model': tag, 'Variant': name, 'Ca': Ca,
                         'Rule': 'p*=Ca/amount',
                         **evaluate(y_test, pt, amount_test, Ca, y_pred=yp)})
    return pd.DataFrame(rows)

print("run_all() ready.")

run_all() ready.


### 5.2 Models: baselines and weighted variants

For each dataset and model I train three variants, all on the natural (un-resampled) training set:

- **baseline** — no cost-sensitivity;
- **class-weight** — class-level weighting (XGBoost `scale_pos_weight`, NN `class_weight`), Scenario A;
- **amount-weight** — example-dependent weights (weight = transaction amount for fraud, $C_a$ for legitimate), Scenario B.

The hyperparameters are tuned once on the baseline (scored on PR-AUC) and reused for the weighted variants, so the only thing that changes between variants is the cost mechanism. Set `DEVICE = 'cpu'` below if no GPU is available.

In [49]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

DEVICE = 'cuda'   # set to 'cpu' if no GPU is available

param_grid = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [3, 4, 5, 6, 8],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2, 0.3],
    'subsample':        [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
}

def tune_xgb(X_tr, y_tr, enable_categorical=False, n_iter=30):
    base = XGBClassifier(eval_metric='aucpr', random_state=42, tree_method='hist',
                         enable_categorical=enable_categorical, device=DEVICE, n_jobs=2)
    search = RandomizedSearchCV(
        base, param_grid, n_iter=n_iter, scoring='average_precision',   # PR-AUC, not recall
        cv=StratifiedKFold(3, shuffle=True, random_state=42),
        random_state=42, n_jobs=2, verbose=1)
    search.fit(X_tr, y_tr)
    print("  best params:", search.best_params_)
    print("  best CV PR-AUC: %.4f" % search.best_score_)
    return search.best_params_

def make_xgb(best, enable_categorical=False, **override):
    p = dict(best); p.update(override)
    return XGBClassifier(eval_metric='aucpr', random_state=42, tree_method='hist',
                         enable_categorical=enable_categorical, device=DEVICE, n_jobs=-1, **p)

def xgb_variants(best, X_tr, y_tr, amount_tr, X_va, X_te, enable_categorical=False):
    """Train baseline, class-weight and amount-weight; return the proba dict for run_all."""
    y_tr = np.asarray(y_tr)
    spw  = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)              # neg/pos ratio
    sw   = np.where(y_tr == 1, np.asarray(amount_tr, dtype=float), CA_PRIMARY)  # example-dependent weights

    m_base = make_xgb(best, enable_categorical);                      m_base.fit(X_tr, y_tr)
    m_cw   = make_xgb(best, enable_categorical, scale_pos_weight=spw); m_cw.fit(X_tr, y_tr)
    m_ew   = make_xgb(best, enable_categorical);                      m_ew.fit(X_tr, y_tr, sample_weight=sw)

    def pv_pt(m): return m.predict_proba(X_va)[:, 1], m.predict_proba(X_te)[:, 1]
    b = pv_pt(m_base); c = pv_pt(m_cw); e = pv_pt(m_ew)
    return {
        'XGB baseline':      {'pv': b[0], 'pt': b[1], 'weighted': False},
        'XGB class-weight':  {'pv': c[0], 'pt': c[1], 'weighted': True},
        'XGB amount-weight': {'pv': e[0], 'pt': e[1], 'weighted': True},
    }

print("XGBoost training helpers ready.")

XGBoost training helpers ready.


In [50]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def numericize_for_nn(X_tr, X_va, X_te, high_card):
    """Frequency-encode the leftover object columns (frequency learned on train); turn bool into int."""
    X_tr, X_va, X_te = X_tr.copy(), X_va.copy(), X_te.copy()
    for col in high_card:
        if col in X_tr.columns:
            freq = X_tr[col].value_counts(normalize=True)
            for Xd in (X_tr, X_va, X_te):
                Xd[col] = Xd[col].map(freq).fillna(0).astype(float)
    for Xd in (X_tr, X_va, X_te):
        bcols = Xd.select_dtypes(include='bool').columns
        Xd[bcols] = Xd[bcols].astype(int)
    return X_tr, X_va, X_te

def build_nn(input_dim):
    m = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),
    ])
    m.compile(optimizer='adam', loss='binary_crossentropy',
              metrics=[keras.metrics.AUC(name='pr_auc', curve='PR')])
    return m

def nn_variants(X_tr, y_tr, amount_tr, X_va, X_te, epochs=20, batch_size=256):
    """Train baseline, class-weight and amount-weight NNs; return the proba dict for run_all."""
    y_tr = np.asarray(y_tr).astype('float32')
    Xtr = X_tr.values.astype('float32'); Xva = X_va.values.astype('float32'); Xte = X_te.values.astype('float32')
    input_dim = Xtr.shape[1]

    spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    sw  = np.where(y_tr == 1, np.asarray(amount_tr, dtype='float32'), CA_PRIMARY).astype('float32')

    def train(class_weight=None, sample_weight=None):
        tf.keras.utils.set_random_seed(42)
        m = build_nn(input_dim)
        m.fit(Xtr, y_tr, validation_split=0.1, epochs=epochs, batch_size=batch_size,
              verbose=0, class_weight=class_weight, sample_weight=sample_weight)
        return m.predict(Xva, verbose=0).ravel(), m.predict(Xte, verbose=0).ravel()

    b = train()
    c = train(class_weight={0: 1.0, 1: float(spw)})
    e = train(sample_weight=sw)
    return {
        'NN baseline':      {'pv': b[0], 'pt': b[1], 'weighted': False},
        'NN class-weight':  {'pv': c[0], 'pt': c[1], 'weighted': True},
        'NN amount-weight': {'pv': e[0], 'pt': e[1], 'weighted': True},
    }

print("Neural-network training helpers ready.")

Neural-network training helpers ready.


#### 5.2.1 IEEE-CIS: train all variants

In [51]:
# IEEE-CIS: cast the leftover high-cardinality object columns to 'category' for XGBoost.
high_card = ['P_emaildomain', 'id_30', 'id_31', 'id_33', 'DeviceInfo']
Xi_tr, Xi_va, Xi_te = X_train.copy(), X_val.copy(), X_test.copy()
for col in high_card:
    if col in Xi_tr.columns:
        cats = pd.concat([Xi_tr[col], Xi_va[col], Xi_te[col]]).astype('category').cat.categories
        for Xd in (Xi_tr, Xi_va, Xi_te):
            Xd[col] = pd.Categorical(Xd[col], categories=cats)

print("Tuning XGBoost on IEEE-CIS ...")
best_xgb_ieee = tune_xgb(Xi_tr, y_train, enable_categorical=True)
var_xgb_ieee  = xgb_variants(best_xgb_ieee, Xi_tr, y_train, amount_train,
                             Xi_va, Xi_te, enable_categorical=True)

print("Training neural networks on IEEE-CIS ...")
Xn_tr, Xn_va, Xn_te = numericize_for_nn(X_train, X_val, X_test, high_card)
var_nn_ieee = nn_variants(Xn_tr, y_train, amount_train, Xn_va, Xn_te)
print("IEEE-CIS variants trained.")

Tuning XGBoost on IEEE-CIS ...
Fitting 3 folds for each of 30 candidates, totalling 90 fits


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [18:59:06] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [18:59:06] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the devic

  best params: {'subsample': 1.0, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 8, 'learning_rate': 0.3, 'colsample_bytree': 0.6}
  best CV PR-AUC: 0.7533


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [19:11:00] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Training neural networks on IEEE-CIS ...


I0000 00:00:1788808269.890501      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14671 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1788808275.998471     170 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


IEEE-CIS variants trained.


#### 5.2.2 ULB: train all variants

In [52]:
# ULB is already fully numeric (V1..V28, Amount, Time), so no categorical handling is needed.
print("Tuning XGBoost on ULB ...")
best_xgb_ulb = tune_xgb(X_ulm_train, y_ulm_train, enable_categorical=False)
var_xgb_ulb  = xgb_variants(best_xgb_ulb, X_ulm_train, y_ulm_train, amount_ulm_train,
                            X_ulm_val, X_ulm_test, enable_categorical=False)

print("Training neural networks on ULB ...")
var_nn_ulb = nn_variants(X_ulm_train, y_ulm_train, amount_ulm_train, X_ulm_val, X_ulm_test)
print("ULB variants trained.")

Tuning XGBoost on ULB ...
Fitting 3 folds for each of 30 candidates, totalling 90 fits


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [19:15:51] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [19:15:51] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the devic

  best params: {'subsample': 0.8, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
  best CV PR-AUC: 0.8375
Training neural networks on ULB ...
ULB variants trained.


## 6. Experimental Results

Every combination of (variant × $C_a$ × decision rule) is evaluated on the held-out **test** set, while the thresholds are chosen on the **validation** set only. The results are collected into one table and saved to `cost_sensitive_results.csv`.

In [53]:
results = pd.concat([
    run_all('IEEE / XGBoost', var_xgb_ieee, y_val,     y_test,     amount_val,     amount_test),
    run_all('IEEE / NN',      var_nn_ieee,  y_val,     y_test,     amount_val,     amount_test),
    run_all('ULB / XGBoost',  var_xgb_ulb,  y_ulm_val, y_ulm_test, amount_ulm_val, amount_ulm_test),
    run_all('ULB / NN',       var_nn_ulb,   y_ulm_val, y_ulm_test, amount_ulm_val, amount_ulm_test),
], ignore_index=True)

results.to_csv('cost_sensitive_results.csv', index=False)
print("Saved cost_sensitive_results.csv  —", results.shape[0], "rows")
results.head(12)


/tmp/ipykernel_58/2846451197.py:55: RuntimeWarning: divide by zero encountered in divide
  thr = np.where(amount > 0, Ca / amount, np.inf)   # very small or zero amounts are never worth declining
/tmp/ipykernel_58/2846451197.py:55: RuntimeWarning: divide by zero encountered in divide
  thr = np.where(amount > 0, Ca / amount, np.inf)   # very small or zero amounts are never worth declining
/tmp/ipykernel_58/2846451197.py:55: RuntimeWarning: divide by zero encountered in divide
  thr = np.where(amount > 0, Ca / amount, np.inf)   # very small or zero amounts are never worth declining
/tmp/ipykernel_58/2846451197.py:55: RuntimeWarning: divide by zero encountered in divide
  thr = np.where(amount > 0, Ca / amount, np.inf)   # very small or zero amounts are never worth declining
/tmp/ipykernel_58/2846451197.py:55: RuntimeWarning: divide by zero encountered in divide
  thr = np.where(amount > 0, Ca / amount, np.inf)   # very small or zero amounts are never worth declining
/tmp/ipykernel_58/28

Saved cost_sensitive_results.csv  — 108 rows


/tmp/ipykernel_58/2846451197.py:55: RuntimeWarning: divide by zero encountered in divide
  thr = np.where(amount > 0, Ca / amount, np.inf)   # very small or zero amounts are never worth declining


,Dataset_Model,Variant,Ca,Rule,Recall,Precision,PR_AUC,FN,FP,Cost,Savings
0,IEEE / XGBoost,XGB baseline,1.0,t=0.5,0.6015,0.9224,0.7972,1647,209,301422.34,0.5284
1,IEEE / XGBoost,XGB baseline,1.0,t*=0.007,0.9506,0.1140,0.7972,204,30531,76117.66,0.8809
2,IEEE / XGBoost,XGB baseline,1.0,p*=Ca/amount,0.9175,0.1402,0.7972,341,23264,50822.89,0.9205
3,IEEE / XGBoost,XGB class-weight,1.0,t=0.5,0.6298,0.8970,0.7767,1530,299,288953.00,0.5479
4,IEEE / XGBoost,XGB class-weight,1.0,t*=0.006,0.9499,0.1032,0.7767,207,34122,74117.01,0.8840
5,IEEE / XGBoost,XGB class-weight,1.0,p*=Ca/amount,0.9199,0.1258,0.7767,331,26414,49767.67,0.9221
6,IEEE / XGBoost,XGB amount-weight,1.0,t=0.5,0.5884,0.8732,0.7504,1701,353,270631.24,0.5766
7,IEEE / XGBoost,XGB amount-weight,1.0,t*=0.008,0.9436,0.1040,0.7504,233,33610,89767.22,0.8595
8,IEEE / XGBoost,XGB amount-weight,1.0,p*=Ca/amount,0.9095,0.1276,0.7504,374,25709,54342.15,0.9150
9,IEEE / XGBoost,XGB baseline,2.5,t=0.5,0.6015,0.9224,0.7972,1647,209,305464.84,0.5221


### 6.1 Best configuration per dataset and model (by Savings, primary $C_a = 2.5$)

In [54]:
primary = results[results['Ca'] == CA_PRIMARY].copy()
best = (primary.sort_values('Savings', ascending=False)
               .groupby('Dataset_Model', as_index=False)
               .first())
print("Best-savings configuration per dataset/model at Ca = %.2f EUR:" % CA_PRIMARY)
best[['Dataset_Model', 'Variant', 'Rule', 'Recall', 'Precision', 'PR_AUC', 'FN', 'FP', 'Cost', 'Savings']]


Best-savings configuration per dataset/model at Ca = 2.50 EUR:


,Dataset_Model,Variant,Rule,Recall,Precision,PR_AUC,FN,FP,Cost,Savings
0,IEEE / NN,NN class-weight,p*=Ca/amount,0.7827,0.1204,0.4968,898,23643,126354.99,0.8023
1,IEEE / XGBoost,XGB class-weight,p*=Ca/amount,0.8689,0.2140,0.7767,542,13193,83438.82,0.8695
2,ULB / NN,NN class-weight,p*=Ca/amount,0.4842,0.4842,0.7077,49,49,2879.43,0.8050
3,ULB / XGBoost,XGB baseline,p*=Ca/amount,0.4842,0.3286,0.8320,49,94,2991.93,0.7974


### 6.2 Baseline vs. cost-sensitive (primary $C_a$): condensed view

In [55]:
# Compact comparison: for each dataset/model, the baseline at 0.5 next to the best cost-sensitive row.
view = primary.copy()
base_mask = (view['Variant'].str.contains('baseline')) & (view['Rule'] == 't=0.5')
baselines = view[base_mask][['Dataset_Model', 'Recall', 'FN', 'FP', 'Savings']]
baselines = baselines.rename(columns={'Recall': 'Recall_base', 'FN': 'FN_base',
                                      'FP': 'FP_base', 'Savings': 'Savings_base'})
compare = best.merge(baselines, on='Dataset_Model', how='left')
compare['FN_reduction'] = compare['FN_base'] - compare['FN']
compare[['Dataset_Model', 'Variant', 'Rule',
         'FN_base', 'FN', 'FN_reduction',
         'Savings_base', 'Savings', 'Recall_base', 'Recall']]

,Dataset_Model,Variant,Rule,FN_base,FN,FN_reduction,Savings_base,Savings,Recall_base,Recall
0,IEEE / NN,NN class-weight,p*=Ca/amount,2743,898,1845,0.2809,0.8023,0.3363,0.7827
1,IEEE / XGBoost,XGB class-weight,p*=Ca/amount,1647,542,1105,0.5221,0.8695,0.6015,0.8689
2,ULB / NN,NN class-weight,p*=Ca/amount,27,49,-22,0.5708,0.8050,0.7158,0.4842
3,ULB / XGBoost,XGB baseline,p*=Ca/amount,24,49,-25,0.7301,0.7974,0.7474,0.4842


### 6.3 Sensitivity analysis over $C_a$

How the best per-model savings change as the administrative cost varies over {1, 2.5, 5} EUR.

In [56]:
sens = (results.sort_values('Savings', ascending=False)
                .groupby(['Dataset_Model', 'Ca'], as_index=False)
                .first())
sens_pivot = sens.pivot(index='Dataset_Model', columns='Ca', values='Savings')
print("Best achievable Savings by Ca (EUR):")
sens_pivot


Best achievable Savings by Ca (EUR):


Ca,1.0,2.5,5.0
Dataset_Model,,,
IEEE / NN,0.8905,0.8023,0.7113
IEEE / XGBoost,0.9221,0.8695,0.8076
ULB / NN,0.8123,0.8050,0.7991
ULB / XGBoost,0.8062,0.7974,0.7890


In [57]:
from sklearn.metrics import confusion_matrix

CM_CA = CA_PRIMARY  # 2.5 EUR

DATASETS = {
    'IEEE / XGBoost': (var_xgb_ieee, y_val,     y_test,     amount_val,     amount_test),
    'IEEE / NN':      (var_nn_ieee,  y_val,     y_test,     amount_val,     amount_test),
    'ULB / XGBoost':  (var_xgb_ulb,  y_ulm_val, y_ulm_test, amount_ulm_val, amount_ulm_test),
    'ULB / NN':       (var_nn_ulb,   y_ulm_val, y_ulm_test, amount_ulm_val, amount_ulm_test),
}

def show_cm(title, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print("  " + title)
    print("                   pred legit   pred fraud")
    print(f"    actual legit   {tn:>10}   {fp:>10}      (TN, FP)")
    print(f"    actual fraud   {fn:>10}   {tp:>10}      (FN, TP)")
    print(f"    -> caught {tp} of {tp+fn} frauds, {fp} false alarms\n")

for tag, (variants, yv, yt, av, at) in DATASETS.items():
    print("=" * 62); print(tag); print("=" * 62)

    # Before: the baseline variant at the default 0.5 cutoff.
    base_name = [n for n in variants if 'baseline' in n][0]
    pt_base = np.asarray(variants[base_name]['pt'])
    show_cm(f"BEFORE - {base_name} @ t=0.5", np.asarray(yt), (pt_base >= 0.5).astype(int))

    # After: the best-savings configuration found earlier (Ca = 2.5).
    row = best[best['Dataset_Model'] == tag].iloc[0]
    vname, rule = row['Variant'], row['Rule']
    info = variants[vname]
    pv, pt = np.asarray(info['pv']), np.asarray(info['pt'])
    if info['weighted']:
        pv, pt = calibrate(pv, yv, pt)
    if rule.startswith('t=0.5'):
        yp = (pt >= 0.5).astype(int)
    elif rule.startswith('t*'):
        t = cost_optimal_threshold(yv, pv, av, CM_CA); yp = (pt >= t).astype(int)
    else:  # p*=Ca/amount
        yp = amount_threshold_predict(pt, at, CM_CA)
    show_cm(f"AFTER  - {vname} + {rule}", np.asarray(yt), yp)

IEEE / XGBoost
  BEFORE - XGB baseline @ t=0.5
                   pred legit   pred fraud
    actual legit       113766          209      (TN, FP)
    actual fraud         1647         2486      (FN, TP)
    -> caught 2486 of 4133 frauds, 209 false alarms

  AFTER  - XGB class-weight + p*=Ca/amount
                   pred legit   pred fraud
    actual legit       100782        13193      (TN, FP)
    actual fraud          542         3591      (FN, TP)
    -> caught 3591 of 4133 frauds, 13193 false alarms

IEEE / NN
  BEFORE - NN baseline @ t=0.5
                   pred legit   pred fraud
    actual legit       113718          257      (TN, FP)
    actual fraud         2743         1390      (FN, TP)
    -> caught 1390 of 4133 frauds, 257 false alarms

  AFTER  - NN class-weight + p*=Ca/amount
                   pred legit   pred fraud
    actual legit        90332        23643      (TN, FP)
    actual fraud          898         3235      (FN, TP)
    -> caught 3235 of 4133 frauds, 236

/tmp/ipykernel_58/2846451197.py:55: RuntimeWarning: divide by zero encountered in divide
  thr = np.where(amount > 0, Ca / amount, np.inf)   # very small or zero amounts are never worth declining
/tmp/ipykernel_58/2846451197.py:55: RuntimeWarning: divide by zero encountered in divide
  thr = np.where(amount > 0, Ca / amount, np.inf)   # very small or zero amounts are never worth declining


In [58]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 200)
cols = ['Dataset_Model','Variant','Ca','Rule','Recall','Precision','FN','FP','Savings']
results[cols].sort_values(['Dataset_Model','Ca','Variant','Rule'])

,Dataset_Model,Variant,Ca,Rule,Recall,Precision,FN,FP,Savings
35,IEEE / NN,NN amount-weight,1.0,p*=Ca/amount,0.8374,0.0718,672,44719,0.8830
34,IEEE / NN,NN amount-weight,1.0,t*=0.011,0.9206,0.0723,328,48827,0.8864
33,IEEE / NN,NN amount-weight,1.0,t=0.5,0.2451,0.6886,3120,458,0.3708
29,IEEE / NN,NN baseline,1.0,p*=Ca/amount,0.8739,0.0822,521,40345,0.8885
28,IEEE / NN,NN baseline,1.0,t*=0.006,0.9519,0.0596,199,62034,0.8461
27,IEEE / NN,NN baseline,1.0,t=0.5,0.3363,0.8440,2743,257,0.2848
32,IEEE / NN,NN class-weight,1.0,p*=Ca/amount,0.8860,0.0808,471,41661,0.8905
31,IEEE / NN,NN class-weight,1.0,t*=0.007,0.9584,0.0604,172,61621,0.8680
30,IEEE / NN,NN class-weight,1.0,t=0.5,0.3092,0.7894,2855,341,0.2370
44,IEEE / NN,NN amount-weight,2.5,p*=Ca/amount,0.7094,0.0970,1201,27296,0.7956


In [59]:
import pandas as pd
df = pd.read_csv('cost_sensitive_results.csv')
print(df[(df.Variant.str.contains('NN baseline')) & (df.Rule=='t=0.5')]
      [['Dataset_Model','PR_AUC']].drop_duplicates().to_string(index=False))

Dataset_Model  PR_AUC
    IEEE / NN  0.5517
     ULB / NN  0.8099
